# Product Agent 실험

상품 후보 식별부터 상품별 문서 검색과 최종 `DomainResult`까지 Product Agent의 전체 흐름을 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 상품 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

Agent와 외부 연결은 한 번만 만들어 재사용한다. 커밋 전에는 검색 원문과 판단 결과 출력을 모두 지운다.

In [ ]:
from pprint import pprint

# Product Agent와 HCX 모델
from pension_agent.agent.model_factory import create_chat_clovax
from pension_agent.agent.product import create_product_agent
from pension_agent.agent.search import SearchService

# CLOVA와 Qdrant 연결 설정
from pension_agent.config import (
    BGE_M3_EMBEDDING_CONFIG,
    MAIN_SUPERVISOR_HCX_CONFIG,
    ClovaStudioConnection,
    QdrantConnection,
)
from pension_agent.retrieval import (
    create_async_qdrant_client,
    create_async_qdrant_retriever,
    create_clova_query_embedder,
    load_product_catalog,
    prewarm_kiwi,
)

## Product Agent 조립

`.env`의 설정으로 HCX, 상품 카탈로그와 Search Service를 준비해 Product Agent에 연결한다.

In [ ]:
# 셀을 다시 실행하면 기존 Qdrant 연결을 먼저 닫는다.
existing_qdrant_client = globals().get("qdrant_client")
if existing_qdrant_client is not None:
    await existing_qdrant_client.close()

# 저장소 루트의 .env에서 연결 정보를 읽는다.
clova_connection = ClovaStudioConnection()
qdrant_connection = QdrantConnection()

# Product Agent가 공용으로 사용할 HCX 모델과 Search Service를 만든다.
model = create_chat_clovax(
    config=MAIN_SUPERVISOR_HCX_CONFIG,
    connection=clova_connection,
)
embedder = create_clova_query_embedder(
    config=BGE_M3_EMBEDDING_CONFIG,
    connection=clova_connection,
)
await prewarm_kiwi()
qdrant_client = create_async_qdrant_client(qdrant_connection)
retriever = create_async_qdrant_retriever(
    qdrant_client,
    connection=qdrant_connection,
)
search_service = SearchService(embedder=embedder, retriever=retriever)

# 같은 카탈로그를 후보 식별과 상품 문서 범위 해석에 사용한다.
product_catalog = load_product_catalog()
product_agent = create_product_agent(
    model=model,
    search_service=search_service,
    catalog=product_catalog,
)

# API key를 제외한 실행 대상만 확인한다.
{
    "qdrant_url": qdrant_connection.url,
    "collection": qdrant_connection.collection,
    "hcx_model": MAIN_SUPERVISOR_HCX_CONFIG.model,
    "embedding_model": BGE_M3_EMBEDDING_CONFIG.model,
    "catalog_size": len(product_catalog.products),
}

## 상품 카탈로그 확인

실험 질문에 사용할 수 있도록 검증된 상품 코드와 이름 일부를 확인한다.

In [ ]:
# 전체 카탈로그가 길어 앞부분만 표시한다.
[
    {
        "product_code": product.product_code,
        "official_name": product.official_name,
        "provider": product.provider,
        "aliases": product.aliases,
    }
    for product in product_catalog.products[:10]
]

## 질문 실행 함수

질문 원문과 Product Agent가 수행할 하나의 판단 목표를 전달한다.

In [ ]:
async def ask_product_agent(*, question: str, objective: str) -> dict[str, object]:
    # Product Agent를 직접 호출해 검증된 DomainResult를 받는다.
    return await product_agent(
        {
            "question": question,
            "objective": objective,
        }
    )

## 상품 하나 분석

카탈로그의 상품 별칭으로 후보를 식별하고 해당 상품 문서의 근거만 사용했는지 확인한다.

In [ ]:
result = await ask_product_agent(
    question="미래에셋 장기성장포커스 1호 주식의 주요 위험과 환매 조건을 알려줘",
    objective="해당 상품의 주요 위험과 유동성 판단",
)
# 판단 상태, 누락 조건과 실제 사용한 문서 근거를 함께 본다.
pprint(result)

In [ ]:
# 선택된 근거가 하나의 상품 문서로 제한됐는지 파일명만 따로 확인한다.
sorted({evidence["source_file_name"] for evidence in result["evidence"]})

## 상품 식별 경계 확인

상품명이 모호하거나 카탈로그에 없을 때 문서를 임의로 선택하지 않고 조건부·미확정 결과를 반환하는지 확인한다.

In [ ]:
# API 제한을 고려해 하나씩 순차 실행한다.
boundary_questions = [
    {
        "question": "한국투자 골드플랜 연금의 위험은 어때?",
        "objective": "분석할 상품을 식별하고 위험 판단",
    },
    {
        "question": "새봄 연금펀드의 위험을 알려줘",
        "objective": "분석할 상품을 식별하고 위험 판단",
    },
]
boundary_results = []
for request in boundary_questions:
    boundary_results.append(await ask_product_agent(**request))

pprint(boundary_results)

## 정리

실험을 마치면 Qdrant 연결을 닫는다. 다시 질문하려면 Product Agent 조립부터 실행한다.

In [ ]:
# 외부 연결을 정리한다.
await qdrant_client.close()
qdrant_client = None
"Product Agent 연결 정리 완료"